<a href="https://colab.research.google.com/github/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution/blob/main/notebooks/05c_adoption_sweep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# What if not every AI image is watermarked or signed?

The evaluation so far is the best case: every AI image carries the watermark and every image carries a C2PA credential. Under that assumption
the watermark alone has a slightly higher F1 than fusion (cross-validation, notebook 07), because it never has to decide a human image.
Real generators don't all watermark (open-source models usually don't) and few images carry credentials, so this asks:
**at what adoption level does fusion overtake the watermark alone?**

It recombines two saved runs of each model on the same test images from notebook 05: the watermarked run and the no-watermark ablation.
For an adoption rate p, each AI image is watermarked with probability p (its CNN score and watermark score come from the matching run),
the watermark detector and the fusion rule are re-applied, and this is repeated 20 times with different random draws.
C2PA adoption is varied the same way (0%, 50%, 100% of images signed).

**No GPU needed**: Runtime > Change runtime type > **CPU**. About 5 minutes.

## 1. Get the code

In [ ]:
%cd /content
!rm -rf repo
!git clone https://github.com/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution.git repo
%cd /content/repo
!pip install -q -e ".[train]"

## 2. Link the saved evaluation results on Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
D = "/content/drive/MyDrive/thesis-data"
!mkdir -p $D/evaluation && rm -rf outputs/evaluation && ln -s $D/evaluation outputs/evaluation
ARCHS = ["resnet50", "efficientnet_b0"]
missing = [f"outputs/evaluation/{a}{s}/per_image.csv" for a in ARCHS for s in ("", "_no_watermark")
           if not os.path.exists(f"outputs/evaluation/{a}{s}/per_image.csv")]
if missing:
    raise FileNotFoundError(f"missing {missing}: run notebook 05 sections 4 and 7 first")
print("found the watermarked and no-watermark runs for both models")

## 3. Run the sweep

In [ ]:
for arch in ARCHS:
    !python scripts/adoption_sweep.py --arch $arch --provenance-adoption 0 0.5 1

## 4. Results
Each line is the mean F1 over 20 random draws (the shaded band is ± 1 sd). **Watermark adoption** on the x-axis is the share of AI images
carrying the watermark. The crossover table lists the adoption levels at which fusion's F1 is at least the watermark-only F1.
Look at coverage next to F1: the watermark alone never decides a human image.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 220)
SHOW = ["clean", "jpeg_q30", "resize_0.5", "resize_0.25"]
STYLE = {"watermark_only": "tab:orange", "cnn_only": "tab:green", "multi_signal": "tab:blue"}
for arch in ARCHS:
    s = pd.read_csv(f"outputs/evaluation/adoption/{arch}/adoption_summary.csv")
    for prov in (1.0, 0.0):
        fig, axes = plt.subplots(1, len(SHOW), figsize=(16, 3.4), sharey=True)
        for ax, cond in zip(axes, SHOW):
            for system, color in STYLE.items():
                d = s[(s.condition == cond) & (s.system == system) & (s.provenance_adoption == prov)].sort_values("watermark_adoption")
                ax.plot(d.watermark_adoption, d.f1_mean, marker="o", color=color, label=system)
                ax.fill_between(d.watermark_adoption, d.f1_mean - d.f1_sd, d.f1_mean + d.f1_sd, color=color, alpha=0.15)
            ax.set_title(cond); ax.set_xlabel("watermark adoption"); ax.set_ylim(0, 1.02)
        axes[0].set_ylabel("F1"); axes[-1].legend(fontsize=8)
        fig.suptitle(f"{arch}: C2PA on {prov:.0%} of images"); plt.tight_layout(); plt.show()
    c = pd.read_csv(f"outputs/evaluation/adoption/{arch}/crossover.csv")
    print(f"{arch}: watermark-adoption levels where fusion F1 >= watermark-only F1")
    display(c[["provenance_adoption", "condition", "levels_where_fusion_at_least_watermark"]])

In [ ]:
cols = ["watermark_adoption", "system", "f1_mean", "f1_sd", "coverage_mean", "recall_mean", "fpr_mean"]
for arch in ARCHS:
    s = pd.read_csv(f"outputs/evaluation/adoption/{arch}/adoption_summary.csv")
    t = s[(s.condition == "jpeg_q30") & (s.provenance_adoption == 1.0) & (s.system != "provenance_only")]
    print(f"{arch} | JPEG q30, C2PA on every image")
    display(t[cols].sort_values(["watermark_adoption", "system"]).round(4))

## 5. Save it
File > Save a copy in GitHub. The summaries are on Drive in `thesis-data/evaluation/adoption/` and are collected by notebook 06.